# Beyond BetterNet: The 100x Frontier
## VM-UNet V2 + FreqMamba + TDA Priors → Target: >0.97 Dice

### πŸŽ― Research Mission
**Surpass BetterNet (2024 SOTA: 0.969 Dice)** using State Space Models + Frequency Domain Intelligence + Topological Awareness

**Intelligent Training Strategy**:
- βœ… Full resolution images (256×256) for feature detail
- βœ… Gradient accumulation (simulates larger batch sizes)
- βœ… All features enabled (FreqMamba + TDA)
- βœ… Smart regularization (prevent overfitting)
- βœ… Learning rate scheduling (warmup + plateau reduction)
- βœ… Seamless training (no kernel crashes)

In [1]:
# 1. GPU DETECTION & SETUP
import tensorflow as tf
import os
import numpy as np
import time
import gc
from glob import glob
from sklearn.model_selection import train_test_split

# Check GPU
print(f"\n{'='*70}")
print("[GPU DETECTION]")
print(f"{'='*70}")

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"βœ… GPU Detected: {[gpu.name for gpu in gpus]}")
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"βœ… Memory growth enabled (dynamic allocation)")
else:
    print(f"β˜ ️ WARNING: No GPU detected!")

print(f"βœ… TensorFlow version: {tf.__version__}")

# Use float32 (mixed precision has issues with AdamW in TF 2.10)
tf.keras.mixed_precision.set_global_policy('float32')
print(f"βœ… Compute policy: {tf.keras.mixed_precision.global_policy().name}")
print(f"{'='*70}\n")


[GPU DETECTION]
βœ… GPU Detected: ['/physical_device:GPU:0']
βœ… Memory growth enabled (dynamic allocation)
βœ… TensorFlow version: 2.10.1
βœ… Compute policy: float32



In [2]:
# 2. IMPORT CUSTOM MODULES
from layers import CBAMModule
from freq_mamba import DualGateFrequencyModule
from mamba import VSSBlock
from vmunet_v2 import vmunet_v2
from tda import topological_loss, extract_topological_features
from tensorflow.keras.optimizers.experimental import AdamW

print("βœ… All custom modules imported successfully")

βœ… All custom modules imported successfully


In [3]:
# 3. INTELLIGENT LOSS FUNCTIONS

def tda_dice_bce_loss(weight_frag=0.08, weight_hole=0.08):
    """Combined Dice + BCE + TDA topological loss - SOTA accuracy"""
    def loss(y_true, y_pred):
        # Clamp for numerical stability
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        
        # Binary Cross-Entropy (fast, punishes confident wrong predictions)
        bce = tf.reduce_mean(tf.keras.losses.binary_crossentropy(y_true, y_pred))
        
        # Dice Loss (handles class imbalance, focuses on overlap)
        smooth = 1e-5
        intersection = tf.reduce_sum(y_true * y_pred, axis=[1, 2, 3])
        union = tf.reduce_sum(y_true, axis=[1, 2, 3]) + tf.reduce_sum(y_pred, axis=[1, 2, 3])
        dice = 1.0 - (2.0 * intersection + smooth) / (union + smooth)
        dice = tf.reduce_mean(dice)
        
        # TDA Topological Loss (penalizes fragments and holes)
        tda_frag, tda_hole = extract_topological_features(y_true, y_pred, threshold=0.5)
        tda = weight_frag * tda_frag + weight_hole * tda_hole
        
        # Final: Weighted combination (0.7 Dice + 0.2 BCE + 0.1 TDA)
        total = 0.7 * dice + 0.2 * bce + 0.1 * tda
        return total
    return loss

def dice_bce_loss():
    """Fast baseline loss without TDA (for debugging)"""
    def loss(y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        bce = tf.reduce_mean(tf.keras.losses.binary_crossentropy(y_true, y_pred))
        
        smooth = 1e-5
        intersection = tf.reduce_sum(y_true * y_pred, axis=[1, 2, 3])
        union = tf.reduce_sum(y_true, axis=[1, 2, 3]) + tf.reduce_sum(y_pred, axis=[1, 2, 3])
        dice = 1.0 - (2.0 * intersection + smooth) / (union + smooth)
        dice = tf.reduce_mean(dice)
        
        return 0.7 * dice + 0.3 * bce
    return loss

@tf.function
def dice_coef(y_true, y_pred):
    """Dice coefficient (higher = better)"""
    smooth = 1e-5
    y_pred_binary = tf.round(y_pred)
    intersection = tf.reduce_sum(y_true * y_pred_binary)
    union = tf.reduce_sum(y_true) + tf.reduce_sum(y_pred_binary)
    return (2.0 * intersection + smooth) / (union + smooth)

@tf.function
def iou_coef(y_true, y_pred):
    """Intersection over Union (higher = better)"""
    smooth = 1e-5
    y_pred_binary = tf.round(y_pred)
    intersection = tf.reduce_sum(y_true * y_pred_binary)
    union = tf.reduce_sum(y_true) + tf.reduce_sum(y_pred_binary) - intersection
    return (intersection + smooth) / (union + smooth)

print("βœ… Loss functions and metrics defined")

βœ… Loss functions and metrics defined


In [4]:
# 4. INTELLIGENT ARCHITECTURE CONFIGURATION
# Gradient accumulation allows us to:
# - Use full 256x256 images (better features)
# - Simulate larger batch sizes with smaller memory footprint
# - Enable all advanced features (FreqMamba + TDA)

# ============================================================
# IMAGE & BATCH CONFIGURATION
# ============================================================
IMG_HEIGHT, IMG_WIDTH = 256, 256  # Full resolution
BATCH_SIZE = 4  # Physical batch size (fits in 12GB RTX 3060)
ACCUM_STEPS = 2  # Accumulate gradients over 2 steps
EFFECTIVE_BATCH = BATCH_SIZE * ACCUM_STEPS  # Effective batch = 8

# ============================================================
# TRAINING HYPERPARAMETERS
# ============================================================
EPOCHS = 200  # More epochs for better convergence
LEARNING_RATE = 1e-4  # Start conservative
WARMUP_EPOCHS = 5  # Warmup period for LR
EARLY_STOP_PATIENCE = 15  # Stop if no improvement for 15 epochs
L2_REGULARIZATION = 1e-4  # L2 penalty for weight decay
DROPOUT_RATE = 0.2  # Prevent overfitting

# ============================================================
# MODEL & FEATURE CONFIGURATION
# ============================================================
EXPERIMENT_NAME = "VMUNet_FullFeatures_GradAccum"
BASE_FILTERS = 24  # Balanced capacity (not minimal, not bloated)
USE_FREQ_MAMBA = True  # βœ… Enable frequency domain learning
USE_TDA_LOSS = True     # βœ… Enable topological awareness

print(f"\n{'='*70}")
print(f"🚀 VMUNET V2 - INTELLIGENT FULL FEATURES")
print(f"{'='*70}")
print(f"\n📋 ARCHITECTURE:")
print(f"   Vision Mamba Encoder: βœ… ENABLED (core SSM backbone)")
print(f"   FreqMamba Module:     βœ… ENABLED (frequency enhancement)")
print(f"   TDA Loss:             βœ… ENABLED (topological awareness)")
print(f"   CBAM Attention:       βœ… ENABLED (channel + spatial)")

print(f"\n πŸ
print(f"   Image Resolution:     {IMG_HEIGHT}Γ—{IMG_WIDTH} (full detail)")
print(f"   Physical Batch Size:  {BATCH_SIZE}")
print(f"   Gradient Accum Steps: {ACCUM_STEPS}")
print(f"   Effective Batch Size: {EFFECTIVE_BATCH}")
print(f"   L2 Regularization:    {L2_REGULARIZATION}")
print(f"   Dropout Rate:         {DROPOUT_RATE}")

print(f"\n βš™οΈ TRAINING SCHEDULE:")
print(f"   Max Epochs:           {EPOCHS}")
print(f"   Learning Rate:        {LEARNING_RATE}")
print(f"   LR Warmup Epochs:     {WARMUP_EPOCHS}")
print(f"   Early Stop Patience:  {EARLY_STOP_PATIENCE} epochs")
print(f"   Optimizer:            AdamW (better generalization)")

print(f"\n πŸ'' STRATEGY:")
print(f"   1. Full images (256Β²) for feature extraction")
print(f"   2. Gradient accumulation simulates batch=8")
print(f"   3. Memory cleanup every 10 steps")
print(f"   4. Learning rate warmup (stable start)")
print(f"   5. ReduceLROnPlateau (rescue from plateaus)")
print(f"   6. Early stopping (prevent overfitting)")
print(f"   7. Checkpoint best model (val_dice monitor)")

print(f"\n 🎯 TARGET: >0.97 Dice (beat BetterNet 0.969)")
print(f"⏱️  Estimated: 18-24 hours (RTX 3060 with all features)")
print(f"{'='*70}\n")

# ============================================================
# BUILD MODEL
# ============================================================
tf.keras.backend.clear_session()
gc.collect()

model = vmunet_v2(
    input_shape=(IMG_HEIGHT, IMG_WIDTH, 3),
    num_classes=1,
    base_filters=BASE_FILTERS,
    use_freq_mamba=USE_FREQ_MAMBA,
    dropout_rate=DROPOUT_RATE,
    l2_reg=L2_REGULARIZATION
)

# Optimizer with AdamW (better generalization + built-in weight decay)
optimizer = AdamW(
    learning_rate=LEARNING_RATE,
    clipnorm=1.0,
    weight_decay=L2_REGULARIZATION
)

# Loss function
loss_fn = tda_dice_bce_loss(weight_frag=0.08, weight_hole=0.08) if USE_TDA_LOSS else dice_bce_loss()

# Compile
model.compile(
    optimizer=optimizer,
    loss=loss_fn,
    metrics=[dice_coef, iou_coef]
)

print(f"βœ… Model built: VM-UNet V2 (FreqMamba + TDA enabled)")
print(f"βœ… Parameters: {model.count_params():,} ({(model.count_params() * 4) / (1024**2):.1f} MB)")
print(f"βœ… Memory safe with intelligent gradient accumulation\n")

SyntaxError: unterminated string literal (detected at line 42) (3467353446.py, line 42)

In [ ]:
# 5. DATA PIPELINE

TRAIN_IMG_PATH = './dataset/TrainDataset/images/*'
TRAIN_MASK_PATH = './dataset/TrainDataset/masks/*'
TEST_IMG_PATH = './dataset/TestDataset/*/images/*'
TEST_MASK_PATH = './dataset/TestDataset/*/masks/*'

def load_dataset_splits():
    train_images = sorted(glob(TRAIN_IMG_PATH))
    train_masks = sorted(glob(TRAIN_MASK_PATH))
    test_images = sorted(glob(TEST_IMG_PATH))
    test_masks = sorted(glob(TEST_MASK_PATH))
    
    if not train_images or not test_images:
        print("[ERROR] Dataset paths not found!")
        return None, None, None, None, None, None
    
    # 90% train, 10% validation
    train_x, val_x, train_y, val_y = train_test_split(
        train_images, train_masks, test_size=0.1, random_state=42
    )
    
    return train_x, val_x, test_images, train_y, val_y, test_masks

def load_and_parse(img_path, mask_path):
    """Load and preprocess image and mask"""
    # Load image
    img = tf.io.read_file(img_path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, [IMG_HEIGHT, IMG_WIDTH])
    img = tf.cast(img, tf.float32) / 255.0
    img = tf.ensure_shape(img, (IMG_HEIGHT, IMG_WIDTH, 3))
    
    # Load mask
    mask = tf.io.read_file(mask_path)
    mask = tf.image.decode_image(mask, channels=1, expand_animations=False)
    mask = tf.image.resize(mask, [IMG_HEIGHT, IMG_WIDTH])
    mask = tf.cast(mask, tf.float32) / 255.0
    mask = tf.ensure_shape(mask, (IMG_HEIGHT, IMG_WIDTH, 1))
    
    return img, mask

def augment(img, mask):
    """Heavy augmentation to prevent overfitting"""
    # Random flip left-right
    if tf.random.uniform(()) > 0.5:
        img = tf.image.flip_left_right(img)
        mask = tf.image.flip_left_right(mask)
    
    # Random flip up-down
    if tf.random.uniform(()) > 0.5:
        img = tf.image.flip_up_down(img)
        mask = tf.image.flip_up_down(mask)
    
    # Random brightness
    img = tf.image.random_brightness(img, 0.2)
    img = tf.clip_by_value(img, 0.0, 1.0)
    
    # Random contrast
    img = tf.image.random_contrast(img, 0.8, 1.2)
    img = tf.clip_by_value(img, 0.0, 1.0)
    
    return img, mask

def create_dataset(img_paths, mask_paths, batch_size=4, augment_data=False):
    """Create tf.data.Dataset with preprocessing"""
    dataset = tf.data.Dataset.from_tensor_slices((img_paths, mask_paths))
    dataset = dataset.map(load_and_parse, num_parallel_calls=tf.data.AUTOTUNE)
    
    if augment_data:
        dataset = dataset.map(augment, num_parallel_calls=tf.data.AUTOTUNE)
        dataset = dataset.shuffle(min(1000, len(img_paths)))
    
    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    
    return dataset

# Load dataset splits
train_x, val_x, test_x, train_y, val_y, test_y = load_dataset_splits()

if train_x:
    print(f"\n{'='*70}")
    print(f"[DATASET SPLITS]")
    print(f"{'='*70}")
    print(f"Training:   {len(train_x)} samples")
    print(f"Validation: {len(val_x)} samples")
    print(f"Test:       {len(test_x)} samples")
    print(f"({'='*70})\n")
    
    # Create datasets
    train_dataset = create_dataset(train_x, train_y, batch_size=BATCH_SIZE, augment_data=True)
    val_dataset = create_dataset(val_x, val_y, batch_size=BATCH_SIZE, augment_data=False)
    test_dataset = create_dataset(test_x, test_y, batch_size=BATCH_SIZE, augment_data=False)
    
    print(f"βœ… Datasets created with batch size = {BATCH_SIZE}")
    print(f"βœ… Augmentation enabled for training data\n")

In [ ]:
# 6. SANITY CHECK
print(f"\n{'='*70}")
print(f"[SANITY CHECK - Before Training]")
print(f"{'='*70}\n")

if not train_x:
    print("ERROR: No training data found!")
else:
    try:
        print("Test 1/5: Loading sample batch...")
        sample = next(iter(train_dataset))
        x_sample, y_sample = sample
        print(f"  βœ… Image shape: {x_sample.shape} (expected: ({BATCH_SIZE}, {IMG_HEIGHT}, {IMG_WIDTH}, 3))")
        print(f"  βœ… Mask shape: {y_sample.shape} (expected: ({BATCH_SIZE}, {IMG_HEIGHT}, {IMG_WIDTH}, 1))")
        
        print("\nTest 2/5: Model forward pass...")
        # Don't compile graph, just check model can handle shapes
        pred = model(x_sample[:1], training=False)
        print(f"  βœ… Output shape: {pred.shape}")
        
        print("\nTest 3/5: Loss computation...")
        loss_val = loss_fn(y_sample[:1], pred)
        print(f"  βœ… Loss value: {float(loss_val):.4f}")
        if not tf.math.is_finite(loss_val):
            print(f"  ⚠️  WARNING: Loss is not finite!")
        else:
            print(f"  βœ… Loss is finite (good)")
        
        print("\nTest 4/5: Gradient computation...")
        with tf.GradientTape() as tape:
            pred = model(x_sample[:2], training=True)
            loss = loss_fn(y_sample[:2], pred)
        grads = tape.gradient(loss, model.trainable_weights)
        print(f"  βœ… Computed {len([g for g in grads if g is not None])} gradients")
        
        print("\nTest 5/5: GPU memory status...")
        gpus = tf.config.list_physical_devices('GPU')
        if gpus:
            print(f"  βœ… GPUs available: {len(gpus)}")
            print(f"  βœ… Memory growth enabled")
        
        print(f"\n{'='*70}")
        print(f"βœ… ALL CHECKS PASSED - READY FOR TRAINING")
        print(f"{'='*70}\n")
        
        del x_sample, y_sample, pred, loss, grads
        gc.collect()
        
    except Exception as e:
        print(f"βœ— ERROR: {type(e).__name__}: {str(e)[:200]}")
        import traceback
        traceback.print_exc()

In [ ]:
# 7. INTELLIGENT TRAINING LOOP WITH GRADIENT ACCUMULATION
import matplotlib.pyplot as plt

os.makedirs('./checkpoints', exist_ok=True)
os.makedirs('./logs', exist_ok=True)
os.makedirs('./results', exist_ok=True)

# Prepare for training
tf.keras.backend.clear_session()
gc.collect()

exp_name = EXPERIMENT_NAME.lower().replace(' ', '_')
checkpoint_path = f"./checkpoints/{exp_name}_best.keras"
csv_path = f"./logs/{exp_name}_history.csv"
plot_path = f"./results/{exp_name}_plots.png"
results_path = f"./results/{exp_name}_results.txt"

# Training variables
best_val_dice = 0.0
best_epoch = 0
patience_count = 0
history = {'epoch': [], 'loss': [], 'dice': [], 'val_loss': [], 'val_dice': [], 'val_iou': [], 'lr': []}

print(f"\n{'='*70}")
print(f"🏋️  STARTING INTELLIGENT TRAINING")
print(f"{'='*70}")
print(f"Gradient Accumulation: {ACCUM_STEPS} steps")
print(f"Effective Batch Size: {EFFECTIVE_BATCH}")
print(f"Dataset: {len(train_x)} training samples")
print(f"{'='*70}\n")

training_start_time = time.time()

try:
    for epoch in range(EPOCHS):
        epoch_start = time.time()
        
        # Learning rate warmup
        if epoch < WARMUP_EPOCHS:
            lr = LEARNING_RATE * (epoch + 1) / WARMUP_EPOCHS
            tf.keras.backend.set_value(optimizer.learning_rate, lr)
        
        # ============================================================
        # TRAINING PHASE - Gradient Accumulation
        # ============================================================
        train_loss = 0.0
        train_dice = 0.0
        train_samples = 0
        accumulated_grads = None
        
        for step, (x_batch, y_batch) in enumerate(train_dataset):
            actual_batch_size = tf.shape(x_batch)[0]
            
            with tf.GradientTape() as tape:
                # Forward pass
                y_pred = model(x_batch, training=True)
                loss = loss_fn(y_batch, y_pred)
                # Scale loss by accumulation steps for proper gradient magnitude
                scaled_loss = loss / ACCUM_STEPS
            
            # Compute gradients
            grads = tape.gradient(scaled_loss, model.trainable_weights)
            
            # Accumulate gradients
            if accumulated_grads is None:
                accumulated_grads = [tf.zeros_like(w) for w in model.trainable_weights]
            
            for i, grad in enumerate(grads):
                if grad is not None:
                    accumulated_grads[i] += grad
            
            # Apply accumulated gradients every ACCUM_STEPS batches
            if (step + 1) % ACCUM_STEPS == 0 or step == len(train_dataset) - 1:
                # Clip accumulated gradients
                clipped_grads, _ = tf.clip_by_global_norm(accumulated_grads, 1.0)
                optimizer.apply_gradients(zip(clipped_grads, model.trainable_weights))
                accumulated_grads = None
            
            # Track metrics
            train_loss += float(loss)
            train_dice += float(dice_coef(y_batch, y_pred))
            train_samples += 1
            
            # Memory cleanup
            if (step + 1) % 10 == 0:
                gc.collect()
            
            del x_batch, y_batch, y_pred, loss, grads, tape
        
        train_loss /= train_samples
        train_dice /= train_samples
        
        # ============================================================
        # VALIDATION PHASE
        # ============================================================
        val_loss = 0.0
        val_dice = 0.0
        val_iou = 0.0
        val_samples = 0
        
        for x_val, y_val in val_dataset:
            y_pred_val = model(x_val, training=False)
            val_loss += float(loss_fn(y_val, y_pred_val))
            val_dice += float(dice_coef(y_val, y_pred_val))
            val_iou += float(iou_coef(y_val, y_pred_val))
            val_samples += 1
            del x_val, y_val, y_pred_val
        
        val_loss /= val_samples
        val_dice /= val_samples
        val_iou /= val_samples
        
        # Get current learning rate
        current_lr = float(tf.keras.backend.get_value(optimizer.learning_rate))
        
        # Track history
        history['epoch'].append(epoch + 1)
        history['loss'].append(train_loss)
        history['dice'].append(train_dice)
        history['val_loss'].append(val_loss)
        history['val_dice'].append(val_dice)
        history['val_iou'].append(val_iou)
        history['lr'].append(current_lr)
        
        # Checkpoint if improved
        epoch_time = (time.time() - epoch_start) / 60
        if val_dice > best_val_dice:
            best_val_dice = val_dice
            best_epoch = epoch + 1
            model.save_weights(checkpoint_path)
            patience_count = 0
            marker = "🏆"
        else:
            patience_count += 1
            marker = "  "
        
        # Learning rate reduction on plateau
        if patience_count > 0 and patience_count % 5 == 0:
            new_lr = current_lr * 0.5
            tf.keras.backend.set_value(optimizer.learning_rate, new_lr)
            print(f"     [LR Reduced] {current_lr:.2e} → {new_lr:.2e}")
        
        # Print progress
        print(f"{marker} Ep {epoch+1:3d}/{EPOCHS} | "
              f"Loss: {train_loss:.4f} | Dice: {train_dice:.4f} | "
              f"Val Dice: {val_dice:.4f} | Val IoU: {val_iou:.4f} | "
              f"LR: {current_lr:.2e} | Time: {epoch_time:.1f}m")
        
        # Early stopping
        if patience_count >= EARLY_STOP_PATIENCE:
            print(f"\n[Early Stopping] No improvement for {EARLY_STOP_PATIENCE} epochs")
            break
        
        # Periodic memory cleanup
        if (epoch + 1) % 5 == 0:
            gc.collect()
    
    training_time = (time.time() - training_start_time) / 3600
    
    print(f"\n{'='*70}")
    print(f"βœ… TRAINING COMPLETED")
    print(f"{'='*70}")
    print(f"Total time: {training_time:.2f} hours")
    print(f"Epochs: {epoch + 1}")
    print(f"Best epoch: {best_epoch} (Val Dice: {best_val_dice:.4f})")
    print(f"{'='*70}\n")
    
except Exception as e:
    print(f"\nβœ— TRAINING FAILED: {type(e).__name__}")
    print(f"Error: {str(e)[:300]}")
    import traceback
    traceback.print_exc()
    raise

In [ ]:
# 8. TEST SET EVALUATION & RESULTS

if train_x:
    print(f"\n{'='*70}")
    print(f"🔬 EVALUATING ON TEST SET")
    print(f"{'='*70}\n")
    
    # Load best weights
    model.load_weights(checkpoint_path)
    
    # Test evaluation
    test_loss = 0.0
    test_dice = 0.0
    test_iou = 0.0
    test_samples = 0
    
    for x_test, y_test in test_dataset:
        y_pred = model(x_test, training=False)
        test_loss += float(loss_fn(y_test, y_pred))
        test_dice += float(dice_coef(y_test, y_pred))
        test_iou += float(iou_coef(y_test, y_pred))
        test_samples += 1
        del x_test, y_test, y_pred
    
    test_loss /= test_samples
    test_dice /= test_samples
    test_iou /= test_samples
    
    # Compare with BetterNet
    betternet_dice = 0.969
    betternet_iou = 0.940
    dice_gain = (test_dice - betternet_dice) * 100
    iou_gain = (test_iou - betternet_iou) * 100
    
    print(f"\n{'='*70}")
    print(f"πŸ
)
    print(f"{'='*70}")
    print(f"\nTest Dice: {test_dice:.4f}")
    if test_dice > betternet_dice:
        print(f"  🏆 SOTA! Beat BetterNet by {dice_gain:+.2f} pp")
    else:
        print(f"  vs BetterNet {betternet_dice:.4f} ({dice_gain:+.2f} pp)")
    
    print(f"\nTest IoU: {test_iou:.4f}")
    if test_iou > betternet_iou:
        print(f"  🏆 SOTA! Beat BetterNet by {iou_gain:+.2f} pp")
    else:
        print(f"  vs BetterNet {betternet_iou:.4f} ({iou_gain:+.2f} pp)")
    
    print(f"\nTest Loss: {test_loss:.4f}")
    
    print(f"\n{'='*70}")
    
    # Save results
    with open(results_path, 'w') as f:
        f.write(f"VMUNET V2 - FULL FEATURES WITH GRADIENT ACCUMULATION\n")
        f.write(f"Experiment: {EXPERIMENT_NAME}\n\n")
        f.write(f"CONFIGURATION:\n")
        f.write(f"  Image Size: {IMG_HEIGHT}x{IMG_WIDTH}\n")
        f.write(f"  Physical Batch: {BATCH_SIZE}\n")
        f.write(f"  Accumulation Steps: {ACCUM_STEPS}\n")
        f.write(f"  Effective Batch: {EFFECTIVE_BATCH}\n")
        f.write(f"  FreqMamba: {'ENABLED' if USE_FREQ_MAMBA else 'DISABLED'}\n")
        f.write(f"  TDA Loss: {'ENABLED' if USE_TDA_LOSS else 'DISABLED'}\n")
        f.write(f"  L2 Regularization: {L2_REGULARIZATION}\n")
        f.write(f"  Dropout: {DROPOUT_RATE}\n\n")
        f.write(f"TEST METRICS:\n")
        f.write(f"  Dice: {test_dice:.4f}\n")
        f.write(f"  IoU:  {test_iou:.4f}\n")
        f.write(f"  Loss: {test_loss:.4f}\n\n")
        f.write(f"VS BETTERNET SOTA:\n")
        f.write(f"  Dice: {test_dice:.4f} vs {betternet_dice:.4f} ({dice_gain:+.2f} pp)\n")
        f.write(f"  IoU:  {test_iou:.4f} vs {betternet_iou:.4f} ({iou_gain:+.2f} pp)\n\n")
        f.write(f"TRAINING DETAILS:\n")
        f.write(f"  Epochs: {epoch + 1}/{EPOCHS}\n")
        f.write(f"  Best Epoch: {best_epoch}\n")
        f.write(f"  Training Time: {training_time:.2f} hours\n")
        f.write(f"  Parameters: {model.count_params():,}\n")
    
    print(f"βœ… Results saved: {results_path}")
    print(f"βœ… Best model: {checkpoint_path}")
    print(f"\n{'='*70}\n")